# 04 · Perfil del top 10 % mensual

¿Qué caracteriza a las vendedoras que Random Forest prioriza? Reconstruimos las predicciones de sus cuatro validaciones temporales, sin entrenar cada modelo con sus filas de validación. Los hiperparámetros ya fueron elegidos sobre estas validaciones: el análisis es descriptivo, no una evaluación final independiente.

Seleccionamos el 10 % de mayor score **dentro de cada mes**, redondeando hacia abajo. Esto difiere del lift por bloque de cuatro meses optimizado anteriormente. La población son vendedoras con compra en el mes observado y al menos un mes activo previo. Una misma persona puede aparecer en varios meses.

`tipo_vendedor` y `departamento` son snapshots maestros y se usan únicamente para describir, no para predecir ni afirmar su estado histórico.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.temporal_optuna import load_raw, temporal_folds, TransactionFeatures, build_model, digest
OUT = ROOT / "reports/optuna_temporal_v1"
raw = load_raw()
protocol = json.loads((OUT / "protocol.json").read_text())
assert digest(ROOT / "data/processed/churn_dataset.csv") == protocol["raw_sha256"]
best = json.loads((OUT / "rf_best.json").read_text())
assert best["params"]["features"] == "all"
oot_start = raw.mes_rank.max() - 3
pool = raw[raw.mes_rank <= oot_start - 7].reset_index(drop=True)


## 1. Reconstruir predicciones de validación

Cada bosque ve solamente el pasado permitido por su fold, con seis meses de separación. Verificamos que su AUC coincida con el trial ganador guardado.

In [2]:
parts = []
for i, (tr, va) in enumerate(temporal_folds(pool)):
    train, valid = pool.iloc[tr], pool.iloc[va]
    transform = TransactionFeatures().fit(train)
    model = build_model("rf", best["params"], train.churn.to_numpy(), threads=4)
    model.fit(transform.transform(train), train.churn)
    p = model.predict_proba(transform.transform(valid))[:, 1]
    auc = roc_auc_score(valid.churn, p)
    assert np.isclose(auc, best["metrics"]["fold_metrics"][i]["auc"], atol=1e-12)
    part = valid.copy()
    part["score"] = p
    part["fold"] = i
    parts.append(part)
validation = pd.concat(parts, ignore_index=True)
print(f"{len(validation):,} observaciones, {validation.id_vendedor.nunique():,} personas, "
      f"{validation.mes_obs.nunique()} meses; AUC de los cuatro folds reproducido.")


4,379 observaciones, 1,506 personas, 16 meses; AUC de los cuatro folds reproducido.


## 2. Seleccionar el decil de cada mes

Ordenamos por score descendente y resolvemos empates por identificador. El score no se interpreta aquí como probabilidad calibrada.

In [3]:
validation["top10_mensual"] = False
monthly = []
for month, group in validation.groupby("mes_obs", sort=True):
    k = max(1, len(group) // 10)
    selected = group.sort_values(["score", "id_vendedor"], ascending=[False, True]).head(k)
    validation.loc[selected.index, "top10_mensual"] = True
    monthly.append({"mes": month.strftime("%Y-%m"), "elegibles": len(group), "priorizadas": k,
                    "churn_real_top": int(selected.churn.sum()),
                    "precision_top": selected.churn.mean(), "prevalencia_mes": group.churn.mean(),
                    "lift_mensual": selected.churn.mean() / group.churn.mean()})
monthly = pd.DataFrame(monthly)
top = validation[validation.top10_mensual]
rest = validation[~validation.top10_mensual]
display(monthly.round(3))
summary = pd.Series({"observaciones_top": len(top), "personas_distintas_top": top.id_vendedor.nunique(),
                     "churn_real_top": top.churn.sum(), "precision_top_agrupada": top.churn.mean(),
                     "churn_resto": rest.churn.mean(), "churn_poblacion": validation.churn.mean(),
                     "lift_mensual_medio": monthly.lift_mensual.mean()})
display(summary)


,mes,elegibles,priorizadas,churn_real_top,precision_top,prevalencia_mes,lift_mensual
0,2023-10,301,30,14,0.467,0.256,1.824
1,2023-11,361,36,25,0.694,0.385,1.804
2,2023-12,309,30,24,0.800,0.369,2.168
3,2024-01,253,25,18,0.720,0.320,2.249
4,2024-02,239,23,10,0.435,0.238,1.823
5,2024-03,242,24,17,0.708,0.289,2.449
6,2024-04,258,25,17,0.680,0.318,2.140
7,2024-05,267,26,19,0.731,0.285,2.567
8,2024-06,250,25,11,0.440,0.224,1.964
9,2024-07,283,28,21,0.750,0.258,2.908


observaciones_top         431.000000
personas_distintas_top    395.000000
churn_real_top            295.000000
precision_top_agrupada      0.684455
churn_resto                 0.266464
churn_poblacion             0.307604
lift_mensual_medio          2.246782
dtype: float64

## 3. Perfil comercial: medianas del top frente al resto

Son asociaciones descriptivas. No prueban causalidad ni explican por sí solas una predicción individual. `compras_hist` cuenta **meses activos previos**, no pedidos.

In [4]:
variables = {
    "compras_hist": "Meses con compra previos",
    "meses_desde_compra_previa": "Meses desde compra anterior",
    "meses_activos_u3": "Meses activos en últimos 3 meses",
    "meses_activos_u12": "Meses activos en últimos 12 meses",
    "n_ped_u3": "Pedidos en últimos 3 meses",
    "n_ped_u12": "Pedidos en últimos 12 meses",
    "monto_u3": "Monto últimos 3 meses",
    "monto_u12": "Monto últimos 12 meses",
    "tend_monto_u3_vs_prev3": "Tendencia monto trimestre actual vs anterior",
}
profile = pd.DataFrame({"top10_mediana": top[list(variables)].median(),
                        "resto_mediana": rest[list(variables)].median()}).rename(index=variables)
display(profile.round(2))


,top10_mediana,resto_mediana
Meses con compra previos,2.00,9.00
Meses desde compra anterior,17.00,1.00
Meses activos en últimos 3 meses,1.00,2.00
Meses activos en últimos 12 meses,1.00,4.00
Pedidos en últimos 3 meses,1.00,2.00
Pedidos en últimos 12 meses,1.00,5.00
Monto últimos 3 meses,270.96,845.90
Monto últimos 12 meses,303.95,1993.26
Tendencia monto trimestre actual vs anterior,1.00,0.28


## 4. Tipo de vendedora según el snapshot disponible

La composición del top debe compararse con la población: una categoría frecuente puede dominar el top simplemente porque domina la base. Estos tipos no entraron al Random Forest.

In [5]:
types = pd.crosstab(validation.tipo_vendedor.fillna("DESCONOCIDO"), validation.top10_mensual)
types = types.reindex(columns=[False, True], fill_value=0).rename(columns={False: "resto", True: "top"})
types["total"] = types.resto + types.top
types["porcentaje_del_top"] = 100 * types.top / types.top.sum()
types["porcentaje_de_base"] = 100 * types.total / types.total.sum()
types["porcentaje_priorizado_del_tipo"] = 100 * types.top / types.total
display(types.round(2))


top10_mensual,resto,top,total,porcentaje_del_top,porcentaje_de_base,porcentaje_priorizado_del_tipo
tipo_vendedor,,,,,,
Asesora,3591,410,4001,95.13,91.37,10.25
Líder,357,21,378,4.87,8.63,5.56


## 5. Guardar resultados descriptivos

Solo exportamos tablas agregadas. No contienen una lista de contactos actuales: los períodos analizados son octubre de 2023 a enero de 2025.

In [6]:
report = ROOT / "reports/perfil_top10_mensual"
report.mkdir(exist_ok=True, parents=True)
monthly.to_csv(report / "metricas_mensuales.csv", index=False)
profile.to_csv(report / "perfil_medianas.csv")
types.to_csv(report / "tipos_vendedora.csv")
summary.to_json(report / "resumen.json", indent=2)
print("Tablas agregadas guardadas en", report.relative_to(ROOT))


Tablas agregadas guardadas en reports/perfil_top10_mensual
